1. get list of zip from VOTER data
2. create df with matched zipcodes to FIPS using the SimpleMaps uscities data (dfZipToFIPS)
1. get relevant FIPS using merged.csv data 
1. get relevant District using zip using

Output:
    * voter.csv

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

### get list of zip from VOTER data

** Reading in raw VOTER data **

In [ ]:
FOLDER2016 = './input'
FILE2016 = os.path.join(FOLDER2016, 'VOTER_Survey_April18_Release1.csv')

IDEO_BASE = ['newsint2_baseline', 'obamaapp_baseline', 'ideo5_baseline', 
             'pid3_baseline', 'vote_generic_baseline', 'cong2012_2_baseline', 
             'PARTYID_SP_baseline', 'polinterest_baseline', 'pol_know_baseline', 
             'straighttic_baseline']

DEMO_BASELINE = ['race_baseline', 'gender_baseline', 
                 'child18_baseline', 'child18num_baseline', 
                 'child18_2016', 
                 'marstat_baseline',
                 'educ_baseline', 'educ_2016', 'educ_2018',
                 'birthyr_baseline', 'student2_baseline', 'employstat2_baseline', 
                 'faminc_baseline', 'faminc_2016', 'faminc_new_2018']

USAGE_BASELINE = ['hourscomputing_baseline', 'intuse_home_baseline', 'intuse_mobile_baseline']

WEIGHTS = ['weight_2016', 'weight_2017', 'weight_panel', 'weight_overall', 'weight_latino', 'weight_18_24']

BASIC = ['case_identifier', 'caseid', 'inputstate_2016', 'izip_2016', 'inputzip_2018']

ELECT_2016 = ['votereg2_2016', 'votereg_f_2016', 'turnout16_2016', 'presvote16post_2016']

SEX2016 = ['ft_fem_2016', 'sexism1_2016', 'sexism2_2016' , 'sexism3_2016', 'sexism4_2016', 
           'sexism5_2016', 'sexism6_2016', 'gender_equality_2016']

IDEO_2016 = ['pid3_2016', 'ideo5_2016', 'newsint_2016']

ELECT_2018 = ['vote18_2018', 'vote18_other_2018', 'trumpapp_2018']

SEX2018 = ['sexism1_2018', 'sexism2_2018' , 'sexism3_2018', 'sexism4_2018', 'sexism5_2018', 'sexism6_2018', 
           'follow_2018', 'alleg_2018', 'sexual_rep_2018', 'sexual_dem_2018', 'attention_2018']

IDEO_2018 = ['pid3_2018', 'ideo5_2018', 'newsint_2018', ]

USECOLS = (BASIC + DEMO_BASELINE + SEX2016 + SEX2018 + ELECT_2016 + ELECT_2018 
           + IDEO_BASE + IDEO_2016 + IDEO_2018 + USAGE_BASELINE + WEIGHTS) 

dfVOTER = pd.read_csv(FILE2016, usecols=USECOLS)
dfVOTER = dfVOTER[USECOLS]
dfVOTER.head()

** Getting zip **
* using izip_2016 and inputzip_2018

Rules:
    * if inputzip_2018 is empty, simply use izip_2016
    * else use inputzip_2018

In [ ]:
dfVOTER['zip'] = np.where(dfVOTER['inputzip_2018'].isnull(), dfVOTER['izip_2016'], dfVOTER['inputzip_2018'])

### create df with matched zipcodes to FIPS using the SimpleMaps uscities data (dfZipToFIPS)

** Read in uscitiesv1.4.csv **

In [ ]:
SIMPLEMAPS_FOLDER = os.path.join('..', '3-parseGeolocation', 'countriesCitiesData')
SIMPLEMAPS_FILE = os.path.join(SIMPLEMAPS_FOLDER, 'uscitiesv1.4.csv')
SIMPLEMAPS_USECOLS = ['county_fips', 'zips']

dfSimplemaps = pd.read_csv(SIMPLEMAPS_FILE, usecols=SIMPLEMAPS_USECOLS)
dfSimplemaps = dfSimplemaps[dfSimplemaps['zips'].notnull()].reset_index()

dfSimplemaps.head()

** Get all zips from VOTER **

In [ ]:
_df = (dfVOTER[dfVOTER['zip'].notnull()]['zip']
       .astype(int))

zip_set = set(_df)

len(zip_set)

In [ ]:
if False:
    dfZipToFIPS = pd.DataFrame(columns=['fips', 'zip'])

    for zipcode in tqdm(zip_set):
        zipcode = str(zipcode)
        iy = len(dfZipToFIPS)
        dfZipToFIPS.at[iy, 'zip'] = zipcode

        for ix in range(len(dfSimplemaps)):
            zips = dfSimplemaps.at[ix, 'zips']
            if zipcode in zips:
                dfZipToFIPS.at[iy, 'fips'] = dfSimplemaps.at[ix, 'county_fips']            

    dfZipToFIPS.to_csv(os.path.join('intermediate', 'ZIPtoFIPS.csv'), index=False)

In [ ]:
dfZipToFIPS = pd.read_csv(os.path.join('intermediate', 'ZIPtoFIPS.csv'))

### get relevant FIPS using merged.csv data 

** Read in merged data **

In [ ]:
MERGEDDATA_FOLDER = os.path.join('..', '7-merge', 'output')
MERGEDDATA_FILE = os.path.join(MERGEDDATA_FOLDER, 'panel.csv')

sampleFIPS = pd.read_csv(MERGEDDATA_FILE, usecols=['fips'])

sampleFIPS = sampleFIPS[sampleFIPS['fips'].notnull()]
sampleFIPS['fips'] = sampleFIPS['fips'].astype(int)
sampleFIPS = set(sampleFIPS['fips'].tolist())
len(sampleFIPS)

In [ ]:
surveyFIPS = set(dfZipToFIPS[dfZipToFIPS['fips'].notnull()]['fips'].tolist())
len(surveyFIPS)

In [ ]:
# No. of FIPS in sample but not in survey
counter = 0
for fip in tqdm(sampleFIPS):
    if fip in surveyFIPS:
        continue
    counter+=1
    
counter

In [ ]:
# No. of FIPS in survey but not in sample
counter = 0
for fip in tqdm(surveyFIPS):
    if fip in sampleFIPS:
        continue
    counter+=1
    
counter

In [ ]:
for ix in tqdm(range(len(dfZipToFIPS))):
    fips = dfZipToFIPS.at[ix, 'fips']
    if fips in sampleFIPS:
        dfZipToFIPS.at[ix, 'inSample'] = 1

In [ ]:
dfZipToFIPS['inSample'].sum()

In [ ]:
len(dfVOTER)

In [ ]:
dfVOTER = dfVOTER.merge(dfZipToFIPS, on=['zip'], how='left')

In [ ]:
len(dfVOTER)

In [ ]:
dfVOTER.to_csv(os.path.join('intermediate', 'voter.csv'), index=False)

### Get district using zip

In [ ]:
stateDict = {'AL': 'Alabama', 
             'AZ': 'Arizona', 
             'AR': 'Arkansas', 
             'CA': 'California', 
             'CO': 'Colorado', 
             'DE': 'Delware at-large', 
             'FL': 'Florida', 
             'GA': 'Georgia', 
             'HI': 'Hawaii', 
             'IA': 'Iowa', 
             'ID': 'Idaho', 
             'IL': 'Illinois', 
             'IN': 'Indiana', 
             'KY': 'Kentucky', 
             'LA': 'Louisiana', 
             'MA': 'Massachusetts', 
             'MD': 'Maryland', 
             'ME': 'Maine', 
             'MI': 'Michigan', 
             'MT': 'Montana at-large', 
             'NC': 'NC', 
             'ND': 'North Dakota at-large', 
             'NE': 'Nebraska', 
             'NH': 'New Hampshire', 
             'NJ': 'New Jersey', 
             'NM': 'New Mexico', 
             'NV': 'Nevada', 
             'NY': 'NY',
             'OH': 'Ohio', 
             'OK': 'Oklahoma', 
             'OR': 'Oregon', 
             'PA': 'Penn', 
             'RI': 'Rhode Island', 
             'SC': 'South Carolina', 
             'SD': 'South Dakota at-large', 
             'TN': 'Tennessee', 
             'TX': 'TX', 
             'UT': 'Utah', 
             'VA': 'Virginia', 
             'VT': 'Vermont at-large', 
             'WA': 'Washington', 
             'WI': 'Wisconsin', 
             'WV': 'West Virginia', 
             'WY': 'Wyoming at-large'}

In [ ]:
ZIP2CONGRESS_FILE = os.path.join('us-zipcodes-congress', 'zccd_hud.csv')

dfZIPCONGRESS = pd.read_csv(ZIP2CONGRESS_FILE)

dfZIPCONGRESS['district_prefix'] = dfZIPCONGRESS['state_abbr'].map(stateDict)
dfZIPCONGRESS['cd'] = dfZIPCONGRESS['cd'].astype('str')
dfZIPCONGRESS['district'] = dfZIPCONGRESS['district_prefix'] + ' ' + dfZIPCONGRESS['cd']
dfZIPCONGRESS['district'] = dfZIPCONGRESS['district'].str.replace('at-large 0', 'at-large')

dfZIPCONGRESS.drop(['state_fips', 'state_abbr', 'cd', 'district_prefix'], axis=1, inplace=True)

dfZIPCONGRESS = dfZIPCONGRESS[dfZIPCONGRESS['district'].notnull()]

dfZIPCONGRESS.drop_duplicates(dfZIPCONGRESS.columns, inplace=True)

dfZIPCONGRESS.head()

In [ ]:
len(dfVOTER)

In [ ]:
dfVOTER = dfVOTER.merge(dfZIPCONGRESS, on='zip', how='left')

In [ ]:
len(dfVOTER)

In [ ]:
dfVOTER.head()

In [ ]:
dfVOTER.to_csv(os.path.join('intermediate', 'voter-district.csv'), index=False)